# Robust04 CHECKPOINT 0.3743: MonoT5-3B Passage-Level Reranking (Run_3)

This notebook is a **dedicated, self-contained reproduction** of the strongest retrieval method discovered in this project (so far): **PARADE-style passage-level MonoT5 reranking** applied to the **fused Run_3** candidate set.

**Checkpoint condition**: MAP improved by **≥ 0.005** over the previous best.

## What this checkpoint captures

- **Candidate set**: fused Run_3 (RM3 + SPLADE++ + SPLADE-v3 + Dense BGE), min-max normalized weighted sum.
- **Reranker**: passage-level MonoT5 with MaxP aggregation.
- **Model**: `cramraj8/duqgen-monot5-3b-robust04-1k`
- **Rerank depth**: `top_n = 1000`
- **Interpolation**: `alpha = 0.3`
- **MAP on judged queries (qids 301–350)**: ≈ **0.3743**

It is designed to be runnable end-to-end to reproduce:

- the **MAP improvement on the 50 judged queries**, and
- the **final run files** for the 199 test queries.

## Why this approach won (intuition + design decisions)

### Why not just BM25 / BM25+RM3?
Robust04 is a newswire/web-era collection with a lot of lexical matching signal, and BM25+RM3 is a strong classical baseline. However, it still misses documents where the relevant content is expressed differently than the query.

### Why score fusion (Run_3) as the base?
We found a consistent improvement by fusing **diverse retrievers**:

- **RM3** (lexical + pseudo-relevance feedback)
- **SPLADE++** and **SPLADE-v3** (learned sparse retrieval; helps with semantic matching while keeping lexical grounding)
- **Dense BGE** (semantic vector retrieval; helps when lexical overlap is weak)

A simple and reliable fusion is to min-max normalize each run’s scores and compute a weighted sum.

### Why neural reranking at all?
Fusion improves the *candidate set*, but it still ranks using shallow similarity signals. A neural reranker is designed to model **query-document relevance** directly, which often yields large improvements when applied to a strong candidate list.

### Why passage-level reranking instead of full-document reranking?
MonoT5 operates over a limited input length. If we stuff the start of a long document into the model, we may miss the relevant span. Passage-level reranking addresses this by:

- splitting each document into overlapping passages,
- scoring each passage for relevance,
- aggregating passage scores back into a document score (we use **MaxP**).

This is directly motivated by PARADE-style aggregation and generally works well for long-document collections.


## Results summary (checkpoint 0.3743)

This checkpoint captures the first time we crossed **MAP ≈ 0.37** on the 50 judged queries using *only* the course-provided title queries.

### Best MAP on the 50 judged queries (qids 301–350)

- **BM25 + RM3** (tuned): MAP ≈ **0.2719**
- **Fusion run_3** (RM3 + SPLADE++ + SPLADE-v3 + Dense): MAP ≈ **0.2997**
- **MonoT5 passage-level reranking** of fused run_3 (inpars-v2 Robust04 model, top-200, MaxP): MAP ≈ **0.3422**
- **MonoT5 passage-level reranking** of fused run_3 (**duqgen Robust04 model**, top-1000, MaxP, alpha=0.3): MAP ≈ **0.3743** (**checkpoint**)

### Sweep evidence (duqgen model, MaxP, fp16)

MAP vs `top_n` (best alpha was **0.3** in the sweeps below):

- 200 → 0.3456
- 300 → 0.3514
- 500 → 0.3649
- 800 → 0.3727
- 1000 → 0.3743

In [ ]:
import os
from collections import defaultdict
from pathlib import Path
from typing import Dict, Iterable, List, Tuple

# IMPORTANT: set before importing Pyserini so the JVM sees it
os.environ.setdefault(
    'JAVA_TOOL_OPTIONS',
    '-Dorg.apache.lucene.store.MMapDirectory.enableMemorySegments=false',
)

import re
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

from pyserini.encode import SpladeQueryEncoder
from pyserini.search.lucene import LuceneHnswDenseSearcher, LuceneImpactSearcher, LuceneSearcher

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device


## Data: queries + qrels

The course-provided split is:

- **Tuning/judged**: first 50 qids
- **Test/submission**: remaining 199 qids

We evaluate MAP only on the 50 judged queries.


In [ ]:
QUERIES_PATH = Path('Files-20260104/queriesROBUST.txt')
QRELS_PATH = Path('Files-20260104/qrels_50_Queries')

def read_queries_tsv(path: Path) -> Dict[str, str]:
    queries: Dict[str, str] = {}
    for line in path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line:
            continue
        qid, query = line.split('	', 1)
        queries[qid] = query
    return queries

def read_qrels(path: Path) -> Dict[str, Dict[str, int]]:
    qrels: Dict[str, Dict[str, int]] = defaultdict(dict)
    for line in path.read_text(encoding='utf-8').splitlines():
        parts = line.strip().split()
        if len(parts) != 4:
            continue
        qid, _, docid, rel = parts
        qrels[qid][docid] = int(rel)
    return qrels

def average_precision(docids: List[str], rels: Dict[str, int]) -> float:
    num_rel = sum(1 for r in rels.values() if r > 0)
    if num_rel == 0:
        return 0.0
    hit = 0
    s = 0.0
    for i, d in enumerate(docids, start=1):
        if rels.get(d, 0) > 0:
            hit += 1
            s += hit / i
    return s / num_rel

def mean_ap(run: Dict[str, List[str]], qrels: Dict[str, Dict[str, int]]) -> float:
    return sum(average_precision(run[qid], qrels[qid]) for qid in run) / len(run)

all_queries = read_queries_tsv(QUERIES_PATH)
train_qids = list(all_queries.keys())[:50]
train_queries = {qid: all_queries[qid] for qid in train_qids}
qrels = read_qrels(QRELS_PATH)

print('queries total:', len(all_queries))
print('train qids:', len(train_queries))
print('qrels qids:', len(qrels))


## Hyperparameter and design choices (checkpointed)

The best performing configuration we converged on for this checkpoint was:

- **Model**: `cramraj8/duqgen-monot5-3b-robust04-1k`

- **Rerank depth**: `top_n = 1000`
  - This is expensive, but we observed consistent MAP gains all the way up to 1000 on judged queries.

- **Interpolation**: `alpha = 0.3`
  - We swept alpha values in `{0, 0.05, 0.1, ..., 0.5}`.
  - `alpha=0.3` was best for this setting (mixing baseline fusion scores with reranker scores within the top-N).

- **Passage splitting**: `passage_chars=1500`, `stride_chars=1200`, `max_passages=8`
  - Overlap (stride < passage length) reduces boundary effects.
  - Capping to 8 passages keeps computation bounded.

- **Aggregation**: `MaxP` (`agg='max'`)
  - If any passage is strongly relevant, the document should be promoted.

- **Precision**: fp16 on GPU
  - Primarily for speed/memory; not expected to materially change MAP.

In [ ]:
# Central configuration (chosen based on sweeps on the 50 judged queries)

# Candidate-set depth
k = 1000

# Run_3 fusion weights (RM3, SPLADE++, SPLADE-v3, Dense)
w_run3 = [0.55, 0.10, 0.15, 0.20]

# Passage-level MonoT5 reranking configuration (checkpoint)
monot5p_model_name = 'cramraj8/duqgen-monot5-3b-robust04-1k'
monot5p_top_n = 1000
monot5p_alpha = 0.3
monot5p_batch_size = 8
monot5p_max_length = 512

monot5p_doc_max_chars = 12000
monot5p_passage_chars = 1500
monot5p_stride_chars = 1200
monot5p_max_passages = 8

monot5p_agg = 'max'       # 'max' (MaxP) or 'avg_topk'
monot5p_avg_topk = 3

use_fp16 = True

print('config:', {
    'k': k,
    'w_run3': w_run3,
    'monot5p_model_name': monot5p_model_name,
    'monot5p_top_n': monot5p_top_n,
    'monot5p_alpha': monot5p_alpha,
    'monot5p_batch_size': monot5p_batch_size,
    'monot5p_max_length': monot5p_max_length,
    'monot5p_doc_max_chars': monot5p_doc_max_chars,
    'monot5p_passage_chars': monot5p_passage_chars,
    'monot5p_stride_chars': monot5p_stride_chars,
    'monot5p_max_passages': monot5p_max_passages,
    'monot5p_agg': monot5p_agg,
    'monot5p_avg_topk': monot5p_avg_topk,
    'use_fp16': use_fp16,
})

def minmax_norm(scores_dict: Dict[str, float]) -> Dict[str, float]:
    if not scores_dict:
        return {}
    vals = list(scores_dict.values())
    mn, mx = min(vals), max(vals)
    if mx - mn < 1e-9:
        return {d: 0.0 for d in scores_dict}
    return {d: (s - mn) / (mx - mn) for d, s in scores_dict.items()}


def fuse_weighted_minmax(
    runs_scores: List[Dict[str, float]],
    weights: List[float],
    depth: int = 1000,
) -> List[Tuple[str, float]]:
    norms = [minmax_norm(rs) for rs in runs_scores]
    docs = set()
    for n in norms:
        docs |= set(n.keys())
    fused_scores: Dict[str, float] = {}
    for d in docs:
        s = 0.0
        for w, n in zip(weights, norms):
            s += w * n.get(d, 0.0)
        fused_scores[d] = s
    ranked = sorted(fused_scores.items(), key=lambda x: (-x[1], x[0]))
    return ranked[:depth]


def ensure_k(ranked: List[str], fallback: List[str], k: int = 1000) -> List[str]:
    if len(ranked) >= k:
        return ranked[:k]
    seen = set(ranked)
    out = list(ranked)
    for d in fallback:
        if d in seen:
            continue
        out.append(d)
        seen.add(d)
        if len(out) >= k:
            break
    return out


def retrieve_scores(searcher, query: str, k: int = 1000) -> Dict[str, float]:
    hits = searcher.search(query, k=k)
    return {h.docid: float(h.score) for h in hits}


rm3 = LuceneSearcher.from_prebuilt_index('robust04')
rm3.set_bm25(0.9, 0.4)
rm3.set_rm3(20, 5, 0.5)

spladepp_encoder = SpladeQueryEncoder('naver/splade-cocondenser-ensembledistil', device=device)
spladepp = LuceneImpactSearcher.from_prebuilt_index('beir-v1.0.0-robust04.splade-pp-ed', spladepp_encoder)

spladev3_encoder = SpladeQueryEncoder('naver/splade-v3-distilbert', device=device)
spladev3 = LuceneImpactSearcher.from_prebuilt_index('beir-v1.0.0-robust04.splade-v3', spladev3_encoder)

dense = LuceneHnswDenseSearcher.from_prebuilt_index(
    'beir-v1.0.0-robust04.bge-base-en-v1.5.hnsw',
    ef_search=1000,
    encoder='BgeBaseEn15',
)

baseline_ranked: Dict[str, List[str]] = {}
baseline_top_scores: Dict[str, Dict[str, float]] = {}

try:
    for i, (qid, query) in enumerate(train_queries.items(), start=1):
        rm3_s = retrieve_scores(rm3, query, k=k)
        pp_s = retrieve_scores(spladepp, query, k=k)
        v3_s = retrieve_scores(spladev3, query, k=k)
        dense_s = retrieve_scores(dense, query, k=k)

        fused = fuse_weighted_minmax([rm3_s, pp_s, v3_s, dense_s], w_run3, depth=k)
        fused_docids = [d for d, _ in fused]
        fused_docids = ensure_k(fused_docids, list(rm3_s.keys()), k=k)
        baseline_ranked[qid] = fused_docids

        baseline_top_scores[qid] = {d: s for d, s in fused[:monot5p_top_n]}

        if i % 10 == 0:
            print(f'built fused candidates {i}/{len(train_queries)}')
finally:
    rm3.close(); spladepp.close(); spladev3.close(); dense.close()

baseline_map = mean_ap(baseline_ranked, qrels)
print('Baseline fused Run_3 MAP (no reranking):', f'{baseline_map:.4f}')

## Passage-level MonoT5 reranking (checkpoint method)

### Reranker model choice
We use the Robust04-tuned MonoT5-3B:

- `cramraj8/duqgen-monot5-3b-robust04-1k`

### Scoring trick (MonoT5 as a classifier)
We follow the standard MonoT5 approach: format input as

`Query: ... Document: ... Relevant:`

and compute a scalar score using the logit difference between generating `true` vs `false` at the first decoding step.

### Passage splitting
Robust04 documents can be long. We split each document into overlapping passages and score each passage independently. Then we aggregate passage scores back to a document score.

- `doc_max_chars = 12000`
- `passage_chars = 1500`
- `stride_chars = 1200`
- `max_passages = 8`

### Aggregation
We use **MaxP** (`agg = 'max'`): if any passage is strongly relevant, the document should be promoted.

### Interpolation with the baseline
We combine baseline and reranker using:

`combined = alpha * baseline_norm + (1-alpha) * reranker_norm`

For this checkpoint, the best setting from our sweeps was **alpha = 0.3** with **top_n = 1000**.

In [ ]:
_TAG_RE = re.compile(r'<[^>]+>')
_WS_RE = re.compile(r'\s+')


def raw_to_text(raw: str) -> str:
    s = _TAG_RE.sub(' ', raw or '')
    s = _WS_RE.sub(' ', s)
    return s.strip()


def chunked(items: List[str], batch_size: int) -> Iterable[List[str]]:
    for i in range(0, len(items), batch_size):
        yield items[i:i + batch_size]


def fetch_doc_texts(searcher: LuceneSearcher, docids: List[str], max_chars: int) -> Dict[str, str]:
    out: Dict[str, str] = {}
    for i, docid in enumerate(docids, start=1):
        try:
            doc = searcher.doc(docid)
            raw = '' if doc is None else (doc.raw() or '')
        except Exception:
            raw = ''
        txt = raw_to_text(raw)
        if max_chars > 0:
            txt = txt[:max_chars]
        out[docid] = txt
        if i % 2000 == 0:
            print(f'fetched {i}/{len(docids)} doc texts')
    return out


def split_passages(text: str, passage_chars: int, stride_chars: int, max_passages: int) -> List[str]:
    t = text or ''
    if passage_chars <= 0:
        return [t]
    if stride_chars <= 0:
        stride_chars = passage_chars
    if max_passages <= 0:
        max_passages = 1
    out: List[str] = []
    i = 0
    while i < len(t) and len(out) < max_passages:
        seg = t[i:i + passage_chars]
        if seg:
            out.append(seg)
        i += stride_chars
    if not out:
        out = ['']
    return out


def compute_monot5_passage_scores(
    tokenizer,
    model,
    true_id: int,
    false_id: int,
    query: str,
    docids: List[str],
    doc_texts: List[str],
    batch_size: int,
    max_length: int,
    passage_chars: int,
    stride_chars: int,
    max_passages: int,
    agg: str,
    avg_topk: int,
) -> Dict[str, float]:
    decoder_start = model.config.decoder_start_token_id
    if decoder_start is None:
        decoder_start = tokenizer.pad_token_id

    ex_docids: List[str] = []
    ex_texts: List[str] = []
    for d, t in zip(docids, doc_texts):
        passages = split_passages(t, passage_chars, stride_chars, max_passages)
        for p in passages:
            ex_docids.append(d)
            ex_texts.append(f'Query: {query} Document: {p} Relevant:')

    doc_to_scores: Dict[str, List[float]] = defaultdict(list)
    with torch.no_grad():
        for batch_docids, batch_text in zip(chunked(ex_docids, batch_size), chunked(ex_texts, batch_size)):
            enc = tokenizer(
                batch_text,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt',
            )
            enc = {k: v.to(device) for k, v in enc.items()}
            decoder_input_ids = torch.full(
                (len(batch_text), 1),
                int(decoder_start),
                dtype=torch.long,
                device=device,
            )
            logits = model(**enc, decoder_input_ids=decoder_input_ids).logits
            step = logits[:, 0, :]
            batch_scores = (step[:, true_id] - step[:, false_id]).detach().cpu().tolist()
            for d, s in zip(batch_docids, batch_scores):
                doc_to_scores[d].append(float(s))

    out: Dict[str, float] = {}
    for d in docids:
        scores = doc_to_scores.get(d, [])
        if not scores:
            out[d] = 0.0
            continue
        if agg == 'avg_topk':
            k_take = max(1, int(avg_topk))
            topk = sorted(scores, reverse=True)[:k_take]
            out[d] = float(sum(topk) / len(topk))
        else:
            out[d] = float(max(scores))
    return out


tokenizer = AutoTokenizer.from_pretrained(monot5p_model_name)
load_kwargs = {}
if use_fp16 and str(device).startswith('cuda'):
    load_kwargs['torch_dtype'] = torch.float16
model = AutoModelForSeq2SeqLM.from_pretrained(monot5p_model_name, **load_kwargs)
model.to(device)
if use_fp16 and str(device).startswith('cuda'):
    model.half()
model.eval()

true_ids = tokenizer.encode('true', add_special_tokens=False)
false_ids = tokenizer.encode('false', add_special_tokens=False)
true_id = true_ids[0]
false_id = false_ids[0]

print('Loaded reranker:', monot5p_model_name)

In [ ]:
# Fetch the document texts needed for reranking (union of top-N docs across queries)
top_docids_all: List[str] = []
for qid in train_qids:
    top_docids_all.extend(baseline_ranked[qid][:monot5p_top_n])
top_docids_all = sorted(set(top_docids_all))
print('Unique docs to fetch for passage reranking:', len(top_docids_all))

bm25 = LuceneSearcher.from_prebuilt_index('robust04')
try:
    doc_texts = fetch_doc_texts(bm25, top_docids_all, max_chars=monot5p_doc_max_chars)
finally:
    bm25.close()

# Run passage-level reranking per query
reranked_run: Dict[str, List[str]] = {}
for i, qid in enumerate(train_qids, start=1):
    query = train_queries[qid]
    top_docids = baseline_ranked[qid][:monot5p_top_n]
    top_texts = [doc_texts.get(d, '') for d in top_docids]

    extra = compute_monot5_passage_scores(
        tokenizer,
        model,
        true_id=true_id,
        false_id=false_id,
        query=query,
        docids=top_docids,
        doc_texts=top_texts,
        batch_size=monot5p_batch_size,
        max_length=monot5p_max_length,
        passage_chars=monot5p_passage_chars,
        stride_chars=monot5p_stride_chars,
        max_passages=monot5p_max_passages,
        agg=monot5p_agg,
        avg_topk=monot5p_avg_topk,
    )

    base = {d: baseline_top_scores[qid].get(d, 0.0) for d in top_docids}
    base_n = minmax_norm(base)
    extra_n = minmax_norm(extra)

    comb = {d: float(monot5p_alpha) * base_n.get(d, 0.0) + (1.0 - float(monot5p_alpha)) * extra_n.get(d, 0.0) for d in top_docids}
    reranked_top = [d for d, _ in sorted(comb.items(), key=lambda x: (-x[1], x[0]))]

    tail = [d for d in baseline_ranked[qid] if d not in set(reranked_top)]
    reranked_run[qid] = (reranked_top + tail)[:k]

    if i % 10 == 0:
        print(f'reranked {i}/{len(train_qids)}')

reranked_map = mean_ap(reranked_run, qrels)
print('Passage-level MonoT5 reranked MAP:', f'{reranked_map:.4f}')


## Generate final submission runs (199 test queries)

For the final submission, we generate:

- `run_1.res`: RM3
- `run_2.res`: fusion (RM3 + SPLADE++ + Dense)
- `run_3.res`: fusion (RM3 + SPLADE++ + SPLADE-v3 + Dense) + **passage-level MonoT5 reranking (top-1000)**

This step uses the project’s `generate_runs.py` script, which mirrors the tuned settings above and writes TREC-formatted run files.

In [ ]:
import subprocess
import sys
import zipfile

# Write to *_m5p.res to avoid overwriting the tracked run_*.res files in the repo
out1 = 'run_1_m5p.res'
out2 = 'run_2_m5p.res'
out3 = 'run_3_m5p.res'

subprocess.run(
    [
        sys.executable,
        'generate_runs.py',
        '--out1', out1,
        '--out2', out2,
        '--out3', out3,
        '--rerank3-monot5-passages',
        '--monot5p-model', 'cramraj8/duqgen-monot5-3b-robust04-1k',
        '--monot5p-alpha', '0.3',
        '--monot5p-top-n', '1000',
        '--monot5p-batch-size', '8',
        '--monot5p-max-length', '512',
        '--monot5p-doc-max-chars', '12000',
        '--monot5p-passage-chars', '1500',
        '--monot5p-stride-chars', '1200',
        '--monot5p-max-passages', '8',
        '--monot5p-agg', 'max',
        '--monot5p-fp16',
    ],
    check=True,
)

(out1, out2, out3)

In [ ]:
# Sanity-check run format (qid coverage, ranks, score monotonicity)
EXPECTED_QIDS = set(map(str, list(range(351, 451)) + list(range(601, 672)) + list(range(673, 701))))

def sanity_check_run(path: str):
    qids = set()
    bad = 0
    line_count = 0
    last_qid = None
    last_rank = 0
    last_score = None
    with Path(path).open('r', encoding='utf-8') as f:
        for line in f:
            line_count += 1
            parts = line.strip().split()
            if len(parts) != 6:
                bad += 1
                continue
            qid, q0, docid, rank, score, tag = parts
            qids.add(qid)
            if q0 != 'Q0':
                bad += 1
            r = int(rank)
            s = float(score)
            if last_qid != qid:
                last_qid = qid
                last_rank = 0
                last_score = None
            if r != last_rank + 1:
                bad += 1
            if last_score is not None and s > last_score + 1e-6:
                bad += 1
            last_rank = r
            last_score = s
    missing = EXPECTED_QIDS - qids
    extra = qids - EXPECTED_QIDS
    return {
        'lines': line_count,
        'unique_qids': len(qids),
        'missing_qids': len(missing),
        'extra_qids': len(extra),
        'bad_checks': bad,
    }

for fname in ['run_1_m5p.res', 'run_2_m5p.res', 'run_3_m5p.res']:
    print(fname, sanity_check_run(fname))

In [ ]:
# Package into a single zip (submission artifact)
# Note: zip uses the required official filenames inside (run_1.res/run_2.res/run_3.res)
zip_name = 'Final_Project_Part_A_runs_m5p.zip'
with zipfile.ZipFile(zip_name, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    z.write('run_1_m5p.res', arcname='run_1.res')
    z.write('run_2_m5p.res', arcname='run_2.res')
    z.write('run_3_m5p.res', arcname='run_3.res')
print('Wrote', zip_name)